# The round trip without a notebook: export from the web app, read in one call

<a href="https://colab.research.google.com/github/theislab/cellucid-python/blob/main/docs/user_guide/r_package/f_notebooks_tutorials/10_round_trip_without_a_notebook.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

Work in the web app at www.cellucid.com with no R session attached,
export what you made, and read every export onto the object with one
call: `cellucid_pull(object, path)`.

**Time:** 10 minutes. The same tutorial in Python:
[Without a notebook](../../python_package/f_notebooks_tutorials/44_round_trip_without_a_notebook.ipynb).

In [ ]:
# A fresh R library (Google Colab's R runtime, a new machine) installs the packages once.
for (package in c("remotes", "SeuratObject")) {
  if (!requireNamespace(package, quietly = TRUE)) install.packages(package)
}
if (!requireNamespace("cellucid", quietly = TRUE)) {
  if ("cellucid" %in% rownames(available.packages())) install.packages("cellucid") else remotes::install_github("theislab/cellucid-r")
}

## The data

SeuratObject's own example, `pbmc_small`: 80 blood cells, 230 genes, a
t-SNE and the clusters `RNA_snn_res.1`. Use your own Seurat or
SingleCellExperiment object in its place.

In [ ]:
library(cellucid)
library(SeuratObject)
data("pbmc_small")
seu <- pbmc_small
seu

## Write the object for the web app

`cellucid_prepare()` writes a Cellucid-ready AnnData (one `.zarr.zip`
file) that the web app opens, with the dataset id stored in it, so a
session saved on it is recognised later.

In [ ]:
seu@misc$cellucid <- list(id = "pbmc-small", name = "PBMC small")
cellucid_prepare(data = seu, out_path = "pbmc_small.zarr.zip", format = "zarr.zip", force = TRUE)

## In the web app

Open https://www.cellucid.com, **Session › Open… › File…**, choose
`pbmc_small.zarr.zip` (the browser reads it; nothing is uploaded), then
make and export:

| Make | Export | File |
|---|---|---|
| a highlight (**Highlighting**, **Confirm**) | **CSV** beside **Clear** | `<page>_cells.csv` |
| a label field (**Copy** `groups`, rename it and its categories) | the field's **Export** | `<field>.csv` |
| a gene set (the gene set editor) | the editor's **Export** | a `.gmt` file |
| everything at once | **Session › Save** | a `.cellucid-session` file |

Put the files in one folder, `cellucid-exports`.

In [ ]:
exports <- "cellucid-exports"
pulled <- cellucid_pull(seu, exports, return_summary = TRUE)
seu <- pulled$data
names(pulled$summary$columns)

`cellucid_pull()` takes one file, several, or a folder (every `.csv`,
`.gmt` and `.cellucid-session` in it):

- a CSV column is written under its own name: `true`/`false` only is a
  highlight (cells the file does not list are `FALSE`), numbers a
  numeric column, anything else a factor; a cell the object does not
  have is refused, by name;
- GMT sets go to `seu@misc$cellucid$gene_sets` (`metadata(sce)$cellucid`);
- a session writes its highlights and fields after checking it was
  saved on this dataset (`misc$cellucid$id`, or `expected_dataset_id =`).

A column the object already has that Cellucid did not write
(`groups`, say) is a conflict: export your own fields, or pass
`on_conflict = "suffix"`.

In [ ]:
head(seu[[]][, names(pulled$summary$columns), drop = FALSE])
pulled$summary$gene_sets

A GMT file is also read on its own:

In [ ]:
for (file in list.files(exports, pattern = "[.]gmt$", full.names = TRUE)) print(cellucid_read_gmt(file))

## Next

- [Subset and re-analyse](12_subset_and_reanalyse.ipynb)
- Python: [Without a notebook](../../python_package/f_notebooks_tutorials/44_round_trip_without_a_notebook.ipynb)